# Project 69 — Local Memory Strategy Benchmark
## Compare Buffer, Summary, Window & Vector Memory

**Stack:** LangChain · Ollama · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Shared Conversation History

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import time, json, shutil

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

conversation = [
    ("user", "Hi, I'm Alice and I work at TechCorp as a data scientist."),
    ("assistant", "Hello Alice! Nice to meet you. Data science at TechCorp sounds exciting."),
    ("user", "I'm building a churn prediction model using XGBoost on 500K customer records."),
    ("assistant", "XGBoost is excellent for churn. What features are you engineering?"),
    ("user", "Usage frequency, last login recency, support ticket count, and plan tier."),
    ("assistant", "Good feature set. Consider adding engagement metrics like session duration."),
    ("user", "Great idea! I'm also adding NPS scores from our quarterly surveys."),
    ("assistant", "NPS is a strong signal. How's the class imbalance in your dataset?"),
    ("user", "About 5% churn — quite imbalanced. I'm using SMOTE and class weights."),
    ("assistant", "Smart approach. Monitor precision-recall, not just accuracy."),
    ("user", "My current AUC is 0.87. Target is 0.92 by Q2."),
    ("assistant", "0.87 is solid. Feature interactions and hyperparameter tuning could help."),
]

test_questions = [
    "What's my name and where do I work?",
    "What model am I using and what's my AUC?",
    "What features am I using for churn prediction?",
    "How am I handling class imbalance?",
]
print(f"Conversation: {len(conversation)} messages")
print(f"Test questions: {len(test_questions)}")

## Step 2 — Implement Memory Strategies

In [ ]:
def full_buffer(messages):
    return "\n".join([f"{role}: {msg}" for role, msg in messages])

def window_memory(messages, window=6):
    recent = messages[-window:]
    return "\n".join([f"{role}: {msg}" for role, msg in recent])

def summary_memory(messages):
    full = "\n".join([f"{role}: {msg}" for role, msg in messages])
    chain = ChatPromptTemplate.from_template(
        "Summarize this conversation, preserving ALL key facts (names, numbers, tools, dates): {conv}"
    ) | llm | StrOutputParser()
    return chain.invoke({"conv": full})

def vector_memory(messages):
    docs = [Document(page_content=f"{role}: {msg}", metadata={"turn": i})
            for i, (role, msg) in enumerate(messages)]
    shutil.rmtree("chroma_mem", ignore_errors=True)
    store = Chroma.from_documents(docs, embeddings, persist_directory="chroma_mem")
    return store

strategies = {
    "full_buffer": lambda: full_buffer(conversation[:-2]),  # exclude test q
    "window_4": lambda: window_memory(conversation[:-2], 4),
    "window_8": lambda: window_memory(conversation[:-2], 8),
    "summary": lambda: summary_memory(conversation[:-2]),
}

# Pre-build contexts
contexts = {}
for name, builder in strategies.items():
    start = time.time()
    contexts[name] = builder()
    elapsed = time.time() - start
    ctx_len = len(contexts[name])
    print(f"  {name:<15} {ctx_len:>5} chars  ({elapsed:.1f}s)")

# Vector memory needs special handling
start = time.time()
vector_store = vector_memory(conversation[:-2])
vector_time = time.time() - start
print(f"  {'vector':15} index built ({vector_time:.1f}s)")

## Step 3 — Evaluate Each Strategy

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on the conversation context. Be specific with facts."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

all_results = []
for q in test_questions:
    print(f"\nQ: {q}")
    for name, context in contexts.items():
        start = time.time()
        answer = qa_chain.invoke({"context": context, "question": q})
        elapsed = time.time() - start
        all_results.append({
            "strategy": name, "question": q[:40],
            "answer": answer[:150], "latency": round(elapsed, 2),
            "context_len": len(context),
        })
        print(f"  [{name}] {answer[:80]}...")

    # Vector memory
    start = time.time()
    relevant = vector_store.similarity_search(q, k=4)
    v_context = "\n".join([d.page_content for d in relevant])
    answer = qa_chain.invoke({"context": v_context, "question": q})
    elapsed = time.time() - start
    all_results.append({
        "strategy": "vector", "question": q[:40],
        "answer": answer[:150], "latency": round(elapsed, 2),
        "context_len": len(v_context),
    })
    print(f"  [vector] {answer[:80]}...")

## Step 4 — Comparison Dashboard

In [ ]:
import pandas as pd

rdf = pd.DataFrame(all_results)
print("MEMORY STRATEGY COMPARISON")
print("=" * 60)
summary = rdf.groupby("strategy").agg({
    "latency": ["mean", "max"],
    "context_len": "mean",
}).round(2)
print(summary.to_string())

print("\nContext efficiency (lower is better):")
for strat in rdf["strategy"].unique():
    sub = rdf[rdf["strategy"] == strat]
    print(f"  {strat:<15} avg_context={sub['context_len'].mean():.0f} chars  avg_latency={sub['latency'].mean():.2f}s")

## What You Learned
- **Four memory strategies** compared head-to-head
- **Tradeoffs**: context length vs. information retention
- **Vector memory** for selective retrieval from long histories
- **Quantitative benchmarking** of memory approaches